# Spatio-Temporal Outbreak & Crisis Detection: High-Performance Spatial Aggregation

In this notebook, we take the optimized dataset from Step 1 and apply **Spatio-Temporal Aggregation** using high-performance libraries.

We will:
1. **Spatial Resolution Selection**: Choose a meaningful Uber H3 resolution.
2. **High-Performance Vectorized Indexing**: Use Polars to map coordinates to H3 hex IDs across multiple CPU cores simultaneously, avoiding slow Pandas loops.
3. **Spatio-Temporal Aggregation**: Roll up the data into a Spatio-Temporal Grid Matrix calculating Event Count, Peak Intensity, and Energy Output.
4. **Spatio-Temporal Zero-Padding**: Generate a continuous grid spine to fill missing dates with structural zeroes for unbroken timelines.

In [28]:
import polars as pl
import h3
import warnings
warnings.filterwarnings('ignore')

## 1. Load Optimized Data with Polars
We use Polars for blazingly fast data manipulation compared to Pandas.

In [29]:
# Load the optimized data using Polars
df = pl.read_csv('optimized_earthquake_data_master.csv')
print(f"Loaded {df.height} records using Polars.")

# Ensure time_bucket is parsed as datetime by providing the exact format string
df = df.with_columns(pl.col('time_bucket_1D').str.to_datetime("%Y-%m-%d %H:%M:%S%z", strict=False))
df.head()

Loaded 9768 records using Polars.


time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,net,id,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,time_bucket_1H,time_bucket_1D,net_ak,net_av,net_ci,net_hv,net_nc,net_nm,net_nn,net_ok,net_pr,net_se,net_tx,net_us,net_uu,net_uw,magType_mb,magType_mb_lg,magType_md,magType_ml,magType_mw,magType_mwb,magType_mwr,magType_mww,type_earthquake,type_mining explosion,status_automatic,status_reviewed
str,f64,f64,f64,f64,str,f64,f64,f64,f64,str,str,str,str,str,f64,f64,f64,f64,str,str,str,str,"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2026-06-11 23:39:21.740000+00:…",19.1566,-65.2976,8.0,4.02,"""md""",29.0,219.0,0.8492,0.38,"""pr""","""pr2026162001""","""2026-06-12T01:49:59.556Z""","""94 km N of Culebra, Puerto Ric…","""earthquake""",1.56,2.5,0.12,23.0,"""reviewed""","""pr""","""pr""","""2026-06-11 23:00:00+00:00""",2026-06-11 00:00:00 UTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
"""2026-06-11 21:32:47.364000+00:…",-18.9647,169.0532,168.418,5.0,"""mb""",46.0,86.0,3.929,0.97,"""us""","""us7000ssct""","""2026-06-11T21:49:04.040Z""","""68 km NNW of Isangel, Vanuatu""","""earthquake""",10.29,8.259,0.08,51.0,"""reviewed""","""us""","""us""","""2026-06-11 21:00:00+00:00""",2026-06-11 00:00:00 UTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
"""2026-06-11 20:37:29.510000+00:…",18.8595,-64.667336,59.47,3.05,"""md""",7.0,253.0,0.4393,0.05,"""pr""","""pr71519603""","""2026-06-11T20:57:17.700Z""","""59 km NNE of Cruz Bay, U.S. Vi…","""earthquake""",2.16,2.19,0.178965,3.0,"""reviewed""","""pr""","""pr""","""2026-06-11 20:00:00+00:00""",2026-06-11 00:00:00 UTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
"""2026-06-11 19:30:08.540000+00:…",34.2386,25.1235,30.561,4.5,"""mb""",70.0,81.0,1.064,0.72,"""us""","""us7000ssbd""","""2026-06-11T20:24:12.040Z""","""85 km S of Pýrgos, Greece""","""earthquake""",7.08,6.227,0.064,71.0,"""reviewed""","""us""","""us""","""2026-06-11 19:00:00+00:00""",2026-06-11 00:00:00 UTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
"""2026-06-11 18:07:55.289000+00:…",-27.8085,-71.502,18.8,4.4,"""mwr""",29.0,170.0,0.885,0.54,"""us""","""us7000ssai""","""2026-06-11T18:27:17.040Z""","""112 km NW of Vallenar, Chile""","""earthquake""",5.57,5.543,0.052,36.0,"""reviewed""","""us""","""us""","""2026-06-11 18:00:00+00:00""",2026-06-11 00:00:00 UTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


## 2. Spatial Resolution Selection & High-Performance Vectorized Indexing
We map each event to an Uber H3 Hexagon. We use **Resolution 4** (approx. 1,770 sq km per hexagon). 
Instead of slow Python row-by-row loops, we use Polars' `map_elements` to distribute the H3 conversion across CPU cores.

In [30]:
RESOLUTION = 4

def get_hex(lat, lon):
    try:
        if hasattr(h3, 'geo_to_h3'):
            return h3.geo_to_h3(lat, lon, RESOLUTION)
        else:
            return h3.latlng_to_cell(lat, lon, RESOLUTION)
    except:
        return None

# Use Polars struct mapping for high performance
df = df.with_columns(
    pl.struct(['latitude', 'longitude'])
    .map_elements(lambda x: get_hex(x['latitude'], x['longitude']), return_dtype=pl.Utf8)
    .alias('hex_id')
)

# Drop any rows where hex conversion failed
df = df.drop_nulls(subset=['hex_id'])

print(f"Unique regions (hexagons) impacted: {df['hex_id'].n_unique()}")
df.select(['latitude', 'longitude', 'hex_id']).head()

Unique regions (hexagons) impacted: 4037


latitude,longitude,hex_id
f64,f64,str
19.1566,-65.2976,"""844cec5ffffffff"""
-18.9647,169.0532,"""849e26bffffffff"""
18.8595,-64.667336,"""844cec3ffffffff"""
34.2386,25.1235,"""843f543ffffffff"""
-27.8085,-71.502,"""84b2035ffffffff"""


## 3. Spatio-Temporal Aggregation
Now that every event has a spatial home (`hex_id`) and temporal home (`time_bucket_1D`), we roll them up into a Spatio-Temporal Grid Matrix. 
We calculate three critical metrics:
- `Event Count`: Total number of shocks/cases.
- `Peak Intensity`: Maximum magnitude.
- `Energy Output`: Average magnitude.

In [31]:
# Perform the GroupBy aggregation
aggregated_df = df.group_by(['time_bucket_1D', 'hex_id']).agg(
    pl.len().alias('event_count'),               # Total number of shocks
    pl.col('mag').max().alias('peak_intensity'), # Maximum magnitude observed
    pl.col('mag').mean().alias('energy_output')  # Average magnitude
)

# Sort by time and event count to see the worst disruptions
aggregated_df = aggregated_df.sort(['time_bucket_1D', 'event_count'], descending=[False, True])

print(f"Aggregated down to {aggregated_df.height} spatio-temporal data blocks.")
aggregated_df.head(10)

Aggregated down to 8065 spatio-temporal data blocks.


time_bucket_1D,hex_id,event_count,peak_intensity,energy_output
"datetime[μs, UTC]",str,u32,f64,f64
2025-12-14 00:00:00 UTC,"""840c6d9ffffffff""",2,3.7,3.5
2025-12-14 00:00:00 UTC,"""840c6ddffffffff""",2,4.2,3.65
2025-12-14 00:00:00 UTC,"""842e713ffffffff""",2,4.9,4.85
2025-12-14 00:00:00 UTC,"""8428303ffffffff""",2,4.03,3.555
2025-12-14 00:00:00 UTC,"""840150dffffffff""",1,4.0,4.0
2025-12-14 00:00:00 UTC,"""84dcda3ffffffff""",1,4.6,4.6
2025-12-14 00:00:00 UTC,"""840c2dbffffffff""",1,3.2,3.2
2025-12-14 00:00:00 UTC,"""84e3627ffffffff""",1,4.6,4.6
2025-12-14 00:00:00 UTC,"""84b3089ffffffff""",1,4.1,4.1


## 4. Spatio-Temporal Zero-Padding (Grid Spine)
To make the dataset structurally complete for ML models, we generate a continuous grid "spine"—a matrix containing every unique active hexagon paired with every calendar date. We join our aggregated data back onto this spine and fill missing gaps with 0.

In [32]:
import datetime

# Cast the datetime column to a strict Date format to match the generated spine
aggregated_df = aggregated_df.with_columns(pl.col("time_bucket_1D").cast(pl.Date))

# 1. Generate every single date in your 6-month window
start_date = aggregated_df["time_bucket_1D"].min()
end_date = aggregated_df["time_bucket_1D"].max()

# Create a sequence of dates
date_range = pl.date_range(start_date, end_date, interval="1d", eager=True).alias("time_bucket_1D")
date_spine = pl.DataFrame(date_range)

# 2. Extract all unique active hexagon IDs
unique_hexes = aggregated_df.select("hex_id").unique()

# 3. Create the Spine: Cross-join dates and hexes to get every possible combination
grid_spine = date_spine.join(unique_hexes, how="cross")

# 4. Join your real data back onto the spine and fill missing days with 0
final_spatial_matrix = (
    grid_spine
    .join(aggregated_df, on=["time_bucket_1D", "hex_id"], how="left")
    .with_columns([
        pl.col("event_count").fill_null(0).cast(pl.Int32),
        pl.col("peak_intensity").fill_null(0.0).cast(pl.Float32),
        pl.col("energy_output").fill_null(0.0).cast(pl.Float32)
    ])
    .sort(["hex_id", "time_bucket_1D"])
)

print(f"Spatio-temporal matrix expanded from {aggregated_df.height} to {final_spatial_matrix.height} rows to fix missing dates.")
final_spatial_matrix.head(15)

Spatio-temporal matrix expanded from 8065 to 726660 rows to fix missing dates.


time_bucket_1D,hex_id,event_count,peak_intensity,energy_output
date,str,i32,f32,f32
2025-12-14,"""8400405ffffffff""",0,0.0,0.0
2025-12-15,"""8400405ffffffff""",0,0.0,0.0
2025-12-16,"""8400405ffffffff""",0,0.0,0.0
2025-12-17,"""8400405ffffffff""",0,0.0,0.0
2025-12-18,"""8400405ffffffff""",0,0.0,0.0
…,…,…,…,…
2025-12-24,"""8400405ffffffff""",0,0.0,0.0
2025-12-25,"""8400405ffffffff""",0,0.0,0.0
2025-12-26,"""8400405ffffffff""",0,0.0,0.0


## 5. Saving the Aggregated Master File
This structurally complete Spatio-Temporal Grid Matrix is ready for our Streamlit dashboard and ML models.

In [33]:
output_file = "aggregated_spatial_matrix.csv"
final_spatial_matrix.write_csv(output_file)
print(f"Saved zero-padded mapping matrix to {output_file}")

Saved zero-padded mapping matrix to aggregated_spatial_matrix.csv
